# Metrics: score distributions & ROC curves

**Kernel:** `conda env:.conda-diffusion` when reading scratch inference pickles.

Batch CLI (all defect × model × mode combos)::

    python analysis/analyze_outputs.py

This notebook is the interactive version of that script / the former `AnalyzeDDIM2DDIM_Outputs.ipynb`.
For overlaying saved ROC JSON files see `CompareROCCurves.ipynb`.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

APP_ROOT = Path("/exp/sbnd/app/users/munjung/anomaly-detection")
sys.path.insert(0, str(APP_ROOT))
sys.path.insert(0, str(APP_ROOT / "analysis"))

from configs.paths import DATA_ROOT, ROC_OUT, FIGURES_OUT, ensure_layout, resolve_inference_dirs
from analyze_outputs import bundle_patch_scores, find_bundles
from sklearn.metrics import auc, roc_curve
import matplotlib.pyplot as plt
import numpy as np

ensure_layout()
plt.style.use(str(APP_ROOT / "analysis" / "presentation.mplstyle"))

DEFECT_TYPE = "coh_noise"
MODEL_TYPE = "-anisotropic"
MODE = "rand2ddpm"
T = 200
SCORE = "rms"

nominal_dir, defect_dir = resolve_inference_dirs(DEFECT_TYPE, MODEL_TYPE, MODE)
print("nominal", nominal_dir, nominal_dir.exists())
print("defect ", defect_dir, defect_dir.exists())


In [ ]:
nom_files = find_bundles(nominal_dir, T, MODE) if nominal_dir.exists() else []
def_files = find_bundles(defect_dir, T, MODE) if defect_dir.exists() else []
print(len(nom_files), "nominal,", len(def_files), "defect bundles")

y_score, y_true = [], []
for p in nom_files:
    s = bundle_patch_scores(p, axis=0, score_mode=SCORE)
    y_score.extend(s); y_true.extend([0] * len(s))
for p in def_files:
    s = bundle_patch_scores(p, axis=0, score_mode=SCORE)
    y_score.extend(s); y_true.extend([1] * len(s))

y_score = np.asarray(y_score, float)
y_true = np.asarray(y_true, int)
print("scores", y_score.shape, "positives", y_true.sum())

if len(y_score) and y_true.min() != y_true.max():
    fpr, tpr, _ = roc_curve(y_true, y_score)
    print("AUC", auc(fpr, tpr))
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].hist(y_score[y_true == 0], bins=40, alpha=0.6, label="nominal", density=True)
    ax[0].hist(y_score[y_true == 1], bins=40, alpha=0.6, label="defect", density=True)
    ax[0].legend(); ax[0].set_title("score distributions")
    ax[1].plot(fpr, tpr); ax[1].plot([0, 1], [0, 1], "k--")
    ax[1].set_title(f"ROC AUC={auc(fpr, tpr):.3f}")
    out = FIGURES_OUT / f"roc_{DEFECT_TYPE}_{MODE}{MODEL_TYPE}_T{T}.pdf"
    fig.savefig(out, bbox_inches="tight")
    print("saved", out)
    plt.show()
else:
    print("Not enough labeled scores — check paths (run on EAF kernel if pickles live on scratch).")
